# Eileen Chang — NemotronLabs RE-USE audio cleanup

Restore the public Harvard recording and export a **full lossless FLAC** and a **smaller AAC/M4A listening copy**. Both contain the complete input. This is an audio-only workflow, with no transcription or published text.

The new wrapper code is MIT-licensed. NVIDIA NemotronLabs RE-USE uses OpenMDW 1.1. The archival recording retains its separate noncommercial terms. Schlesinger Library has granted noncommercial reuse of the recording with attribution. See the README for terms and source credits.

Choose **Runtime → Change runtime type → T4 GPU**. Free GPU availability and session lifetime vary. Google Drive is not mounted. The original model completed the recording on a T4. NemotronLabs sample comparisons have passed; this notebook will run the full recording using the new model.

## 1. Load the source code

The notebook downloads the cleanup code from the public GitHub repository. Set `REPOSITORY_URL` to an empty string only if you prefer to upload a source ZIP created with `python scripts/package_source.py`.

In [ ]:
from pathlib import Path
import json, subprocess, sys, zipfile
from google.colab import files

REPOSITORY_URL = "https://github.com/jianershi/eileen-chang-clean-up.git"
PROJECT_ROOT = Path('/content/eileen-chang-clean-up')
if not (PROJECT_ROOT / 'pyproject.toml').is_file():
    if REPOSITORY_URL:
        if not REPOSITORY_URL.startswith('https://'):
            raise ValueError('Use a public HTTPS repository URL.')
        subprocess.run(['git', 'clone', '--', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
    else:
        print('Upload eileen-chang-clean-up-source.zip (code only).')
        uploaded = files.upload()
        archives = [name for name in uploaded if name.endswith('.zip')]
        if len(archives) != 1:
            raise ValueError('Upload exactly one source ZIP.')
        with zipfile.ZipFile(archives[0]) as archive:
            for item in archive.infolist():
                target = (Path('/content') / item.filename).resolve()
                if not target.is_relative_to(PROJECT_ROOT):
                    raise ValueError('Unexpected source archive path.')
            archive.extractall('/content')
        del uploaded
assert (PROJECT_ROOT / 'scripts/setup_gpu.py').is_file(), 'Missing project source.'
profile = (PROJECT_ROOT / 'src/eileen_cleanup/profile.py').read_text()
assert '31f82f2efad1095034492f236e8dac9239ba713b' in profile, 'An older project is loaded; start a fresh Colab runtime.'
print('Project:', PROJECT_ROOT)
print('See README.md for usage, terms and source credits.')

## 2. Install GPU dependencies

Run in a fresh GPU session. The original run used Torch 2.10.0 / CUDA 13.0, causal-conv1d 1.7.0 and mamba-ssm 2.3.2.post1. Setup selects official native wheels matching the active Python, Torch, CUDA, ABI and architecture, and checks a native CUDA operation. Unsupported runtimes stop without launching a source build.

If torch was previously imported in your notebook, restart the runtime after installation before continuing. Processing below uses a fresh subprocess.

In [ ]:
subprocess.run([sys.executable, str(PROJECT_ROOT / 'scripts/setup_gpu.py'), '--install-torch'], check=True)
subprocess.run(['ffmpeg', '-version'], check=True, stdout=subprocess.DEVNULL)
subprocess.run(['ffprobe', '-version'], check=True, stdout=subprocess.DEVNULL)

## 3. Supply your original audio

Obtain your own complete copy from the [Harvard player](https://mps.lib.harvard.edu/sds/audio/503996508), following the library's access procedures. Keep the original for comparison. Any audio encoding FFmpeg can read is accepted; no particular download's byte size or checksum is required.

Upload through Colab's Files sidebar and set `SOURCE` to its path, or use the prompt below. Ensure the input covers the material you intend to restore. The current full archival source is approximately 105 minutes and includes a later seminar. This workflow preserves the entire input.

In [ ]:
SOURCE = Path('/content/recording.ext')  # Set to your own complete source file.
WORK_DIR = Path('/content/reuse_nemotron_work')
OUTPUT_DIR = Path('/content/reuse_nemotron_outputs')
if not SOURCE.is_file():
    print('Upload exactly one complete original audio file. Large uploads may take time.')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one source file, or set SOURCE to a file already uploaded.')
    SOURCE = Path('/content') / next(iter(uploaded))
    del uploaded
print('Source:', SOURCE.name)
print('Work directory:', WORK_DIR)
# /content is temporary. To survive VM deletion, explicitly choose persistent
# storage or back up the whole work directory. Drive is not auto-mounted.

## 4. Run or resume restoration

The default is full float32 with 20-second cores, 2-second context and one-second crossfade margins. Internal input fingerprints bind checkpoints to your file; they do not prescribe a particular source encoding. Valid completed chunks are reused.

Processing took roughly real time on the historical T4; completion time and free GPU availability are not guaranteed. If GPU memory is exhausted, set `CORE_SECONDS = 10` and choose a **new** `WORK_DIR`. After all GPU chunks complete, `finalize` can join/export on a CPU. Existing final files are protected; use `--overwrite` deliberately if re-exporting.

In [ ]:
CORE_SECONDS = 20
subprocess.run([sys.executable, '-m', 'eileen_cleanup', 'restore', str(SOURCE),
                '--work-dir', str(WORK_DIR), '--output-dir', str(OUTPUT_DIR),
                '--core-seconds', str(CORE_SECONDS)], cwd=PROJECT_ROOT, check=True)

## 5. Verify and download

Both formats cover the entire input. Complete decoding checks sample values, headroom and duration. The ZIP includes FLAC, M4A, validation metadata and the README with terms and source credits. It excludes input audio, published text, recognition data and model weights.

Download before leaving the session. To resume after a VM reset, also back up the complete work directory separately; it is not part of this results ZIP.

In [ ]:
MASTER = OUTPUT_DIR / 'NVIDIA_NemotronLabs_REUSE_full_recording.flac'
LISTENING_COPY = OUTPUT_DIR / 'NVIDIA_NemotronLabs_REUSE_full_recording.m4a'
reports = {}
for path in (MASTER, LISTENING_COPY):
    result = subprocess.run([sys.executable, '-m', 'eileen_cleanup', 'verify', str(path)],
                            cwd=PROJECT_ROOT, check=True, capture_output=True, text=True)
    reports[path.name] = json.loads(result.stdout)
    print(json.dumps(reports[path.name], indent=2))
verification = OUTPUT_DIR / 'Verification.json'
verification.write_text(json.dumps(reports, indent=2) + '\n')
archive_path = Path('/content/Eileen_Chang_NemotronLabs_enhanced_audio.zip')
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_STORED) as archive:
    for path in (MASTER, LISTENING_COPY, LISTENING_COPY.with_suffix('.json'),
                 OUTPUT_DIR / 'Restoration_metadata.json', verification):
        archive.write(path, path.name)
    archive.write(PROJECT_ROOT / 'README.md', 'README.md')
    archive.write(PROJECT_ROOT / 'LICENSE', 'LICENSE')
print('Results ready:', archive_path.name)
files.download(str(archive_path))